# EVT-MinHash Distribution Verification and Bootstrap Baseline for Short Text Documents

## Overview

This notebook demonstrates the experiment that empirically verifies whether MinHash signature minima follow Gumbel or Weibull distributions for short text documents.

### Demo Configuration (Minimal Values):
- 32 hash functions (vs 128 in full experiment)
- 100 bootstrap resamples (vs 1000)
- 20 document pairs (vs 1000)
- 1 dataset (tweets only)

In [ ]:
# Install dependencies - aii-colab pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Always install (not on Colab by default)
_pip('loguru')

# Only install if NOT on Colab (match Colab versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'seaborn==0.13.2')


In [ ]:
from loguru import logger
import json, sys, os, gc, time, random, hashlib
import numpy as np
from typing import List, Tuple, Dict, Optional, Any
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gumbel_l, kstest, anderson

# Configure logger
logger.remove()
logger.add(sys.stdout, level='INFO', format='{time:HH:mm:ss}|{level:<7}|{message}')
print('Imports complete')

In [ ]:
# Data loading - GitHub URL with local fallback
GITHUB_DATA_URL = 'https://raw.githubusercontent.com/AMGrobelnik/ai-invention-1ba2b3-why-extreme-value-theory-fails-for-minha/main/round-1/experiment-1/demo/mini_demo_data.json'

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    raise FileNotFoundError('Could not load mini_demo_data.json')

data = load_data()
print(f'Loaded data with {len(data["datasets"])} datasets')
for name, docs in data['datasets'].items():
    print(f'  {name}: {len(docs)} documents')

In [ ]:
# Configuration - ABSOLUTE MINIMUM values for demo
NUM_HASHES = 32
K_SHINGLE = 3
SHINGLE_TYPE = 'char'
NUM_BOOTSTRAP = 100
CONFIDENCE_LEVEL = 0.95
NUM_DOCUMENT_PAIRS = 20
SHINGLE_COUNT_RANGES = [(10, 30)]
DATASETS = ['tweets']
RANDOM_SEED = 42

print('Config: NUM_HASHES={}, NUM_BOOTSTRAP={}'.format(NUM_HASHES, NUM_BOOTSTRAP))

## MinHash Implementation

This section implements the MinHash algorithm for Jaccard similarity estimation.

In [ ]:
class MinHash:
    def __init__(self, num_hashes=128, seed=42):
        self.num_hashes = num_hashes
        self.seed = seed
        self.hash_functions = [lambda x, s=self.seed+i*1000: self._hash(x, s) for i in range(num_hashes)]

    def _hash(self, shingle, seed):
        return int(hashlib.md5(f'{seed}_{shingle}'.encode()).hexdigest(), 16)

    def _get_shingles(self, text, k=3, shingle_type='char'):
        text = text.lower()
        if len(text) < k: return {text}
        return set(text[i:i+k] for i in range(len(text)-k+1))

    def compute_signature(self, text, k=3, shingle_type='char'):
        shingles = self._get_shingles(text, k, shingle_type)
        if not shingles: return [0]*self.num_hashes, 0
        sig = [min(f(shingle) for shingle in shingles) for f in self.hash_functions]
        return sig, len(shingles)

    def jaccard_similarity(self, sig1, sig2):
        return sum(1 for a,b in zip(sig1,sig2) if a==b) / len(sig1)

    def exact_jaccard(self, text1, text2, k=3, shingle_type='char'):
        s1, s2 = self._get_shingles(text1,k,shingle_type), self._get_shingles(text2,k,shingle_type)
        if not s1 and not s2: return 1.0
        union = s1 | s2
        return len(s1 & s2) / len(union) if union else 1.0

minhash = MinHash(num_hashes=NUM_HASHES, seed=RANDOM_SEED)
print('MinHash ready')

## Bootstrap Confidence Intervals

This section implements bootstrap confidence intervals for Jaccard similarity.

In [ ]:
class BootstrapCI:
    def __init__(self, num_bootstrap=1000, random_seed=42):
        self.num_bootstrap = num_bootstrap
        self.rng = random.Random(random_seed)

    def compute_bootstrap_ci(self, doc1, doc2, minhash_obj, k=3, shingle_type='char', confidence=0.95):
        shingles1 = list(minhash_obj._get_shingles(doc1, k, shingle_type))
        shingles2 = list(minhash_obj._get_shingles(doc2, k, shingle_type))
        if not shingles1 or not shingles2: return 0.0, 1.0, 0.0
        sims = []
        for _ in range(self.num_bootstrap):
            s1 = set(self.rng.choice(shingles1) for _ in shingles1)
            s2 = set(self.rng.choice(shingles2) for _ in shingles2)
            u = len(s1 | s2)
            sims.append(len(s1 & s2) / u if u else 1.0)
        alpha = 1 - confidence
        return np.percentile(sims, [(alpha/2)*100, (1-alpha/2)*100]).tolist() + [np.mean(sims)]

bootstrap = BootstrapCI(num_bootstrap=NUM_BOOTSTRAP, random_seed=RANDOM_SEED)
print('BootstrapCI ready')

## Distribution Fitting (Gumbel and Weibull)

Fit Gumbel and Weibull distributions to MinHash signature minima.

In [ ]:
def fit_gumbel(data):
    normalized = (data - data.min()) / (data.max() - data.min() + 1e-10)
    try:
        loc, scale = gumbel_l.fit(normalized, floc=0)
        ks = kstest(normalized, lambda x: gumbel_l.cdf(x, loc=loc, scale=scale))
        return {'dist': 'gumbel', 'loc': loc, 'scale': scale, 'ks_pvalue': ks.pvalue, 'ks_stat': ks.statistic}
    except: return None

def fit_weibull(data):
    from scipy.stats import weibull_min
    normalized = (data - data.min()) / (data.max() - data.min() + 1e-10)
    try:
        shape, loc, scale = weibull_min.fit(normalized, floc=0)
        ks = kstest(normalized, lambda x: weibull_min.cdf(x, shape, loc=loc, scale=scale))
        return {'dist': 'weibull', 'shape': shape, 'loc': loc, 'scale': scale, 'ks_pvalue': ks.pvalue, 'ks_stat': ks.statistic}
    except: return None

print('Distribution fitting functions ready')

## Run the Experiment

Run the main experiment with the demo data.

In [ ]:
# Run experiment
results = {'fit': [], 'bootstrap': []}
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

for dataset_name in DATASETS:
    docs = data['datasets'].get(dataset_name, [])
    print(f'Processing {dataset_name} ({len(docs)} docs)')
    
    for shingle_range in SHINGLE_COUNT_RANGES:
        min_s, max_s = shingle_range
        filtered = []
        for doc in docs:
            _, cnt = minhash.compute_signature(doc, K_SHINGLE, SHINGLE_TYPE)
            if min_s <= cnt <= max_s: filtered.append(doc)
        print(f'  Range {min_s}-{max_s}: {len(filtered)} docs')
        
        if len(filtered) < 5: continue
        
        # Get MinHash minima
        minima = np.array([min(minhash.compute_signature(d, K_SHINGLE, SHINGLE_TYPE)[0]) for d in filtered])
        
        # Fit distributions
        g_result = fit_gumbel(minima)
        w_result = fit_weibull(minima)
        
        best = g_result if (g_result and (not w_result or g_result['ks_pvalue'] > w_result['ks_pvalue'])) else w_result
        if best:
            best['dataset'] = dataset_name
            best['range'] = shingle_range
            results['fit'].append(best)
            print(f'  Best fit: {best["dist"]} (p={best["ks_pvalue"]:.4e})')

print(f'Experiment complete. {len(results["fit"])} fits done.')

In [ ]:
# Results Summary
print('='*60)
print('RESULTS')
print('='*60)

print('\n1. DISTRIBUTION FIT RESULTS:')
for r in results['fit']:
    print(f'  {r["dataset"]} {r["range"]}: {r["dist"]} fit (p={r["ks_pvalue"]:.2e})')
    status = 'GOOD FIT' if r['ks_pvalue'] >= 0.05 else 'POOR FIT'
    print(f'    -> {status}')

print('\n2. OUTPUT FILES:')
print('  - mini_demo_data.json: Demo dataset')
print('  - code_demo.ipynb: This notebook')

print('\nExperiment demonstration complete!')

In [ ]:
# Visualization - MinHash Signature Minima Distribution
import matplotlib.pyplot as plt
import numpy as np

# Get MinHash minima for tweets
minhash_viz = MinHash(num_hashes=NUM_HASHES, seed=RANDOM_SEED)
docs = data['datasets']['tweets']
minima = []
for doc in docs:
    sig, _ = minhash_viz.compute_signature(doc, K_SHINGLE, SHINGLE_TYPE)
    minima.append(min(sig))

print(f'MinHash minima: {len(minima)} values')
print(f'Min: {min(minima)}, Max: {max(minima)}')

# Plot histogram
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.hist(minima, bins=20, edgecolor='black', alpha=0.7, color='skyblue')
plt.xlabel('MinHash Signature Minimum')
plt.ylabel('Frequency')
plt.title('Distribution of MinHash Minima (Tweets)')

# Plot QQ plot vs uniform distribution
plt.subplot(1, 2, 2)
sorted_data = np.sort(minima)
n = len(sorted_data)
theoretical_quantiles = np.linspace(0, 1, n)
plt.scatter(theoretical_quantiles, sorted_data, alpha=0.6, s=30)
plt.plot([0, 1], [min(sorted_data), max(sorted_data)], 'r--', label='Linear')
plt.xlabel('Theoretical Quantiles')
plt.ylabel('Sample Quantiles')
plt.title('QQ Plot (vs Uniform)')
plt.legend()

plt.tight_layout()
plt.show()

print('Visualization complete!')
